# JIT Kernel Speedup Benchmark

Compares unpacked (Python for-loops) vs packed (JAX JIT) kernel performance.

- **Data**: 200 rows, 10 columns, all 5 types (2 continuous, 2 categorical, 2 binary, 1 ordinal, 2 cyclic, 1 continuous)
- **Per-kernel timing**: row assignments, column hypers, CRP alphas
- **Full sweep comparison**: 3 sweeps unpacked vs packed
- **Cyclic-only benchmark**: Von Mises type-specialized fast path with 10 cyclic columns

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import time

import jax

from benchmarks.utils import create_results_dir, detect_platform, save_metrics
from crosscat import (
    ColumnType,
    generate_crosscat_data,
    initialize,
    log_joint,
    pack_state,
    packed_gibbs_sweep,
    packed_transition_column_hypers,
    packed_transition_crp_alphas,
    packed_transition_row_assignments,
)
from crosscat.gibbs import (
    gibbs_sweep,
    transition_column_hypers,
    transition_crp_alphas,
    transition_row_assignments,
)

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Generate Benchmark Data

In [ ]:
key = jax.random.key(42)
column_types = [
    ColumnType.CONTINUOUS,
    ColumnType.CONTINUOUS,
    ColumnType.CATEGORICAL,
    ColumnType.CATEGORICAL,
    ColumnType.BINARY,
    ColumnType.BINARY,
    ColumnType.ORDINAL,
    ColumnType.CYCLIC,
    ColumnType.CYCLIC,
    ColumnType.CONTINUOUS,
]
result = generate_crosscat_data(
    key, 200, column_types, n_views=3, n_clusters=3, cluster_separation=5.0
)
data = result["data"]

# Initialize state
k1, k2 = jax.random.split(jax.random.key(0))
state = initialize(k1, data, column_types).state
state = gibbs_sweep(k2, state, data, n_sweeps=3)
packed = pack_state(state)

print(
    f"Data: {data.shape}, State: {state.n_views} views, "
    f"Log joint: {float(log_joint(state, data)):.2f}"
)


def time_fn(fn, *args, n_runs=3, **kwargs):
    """Time a function, returning (mean_seconds, result)."""
    result = fn(*args, **kwargs)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        if hasattr(result, "column_assignments"):
            result.column_assignments.block_until_ready()
        elif hasattr(result, "view_row_assignments"):
            result.view_row_assignments.block_until_ready()
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return sum(times) / len(times), result

## 3. Per-Kernel Benchmarks

Compare each kernel individually: row assignments, column hypers, CRP alphas.

In [ ]:
results = {"platform": platform}
key = jax.random.key(123)
total_orig, total_packed = 0.0, 0.0

print("Kernel benchmarks (mean of 3 runs):")
print("-" * 70)

# Row assignments
t_o, _ = time_fn(transition_row_assignments, key, state, data)
t_p, _ = time_fn(packed_transition_row_assignments, key, packed, data)
print(
    f"  {'row_assignments':25s}  orig: {t_o:.4f}s  packed: {t_p:.4f}s  {t_o / max(t_p, 1e-9):.1f}x"
)
total_orig += t_o
total_packed += t_p
results["row_assignments_orig"] = t_o
results["row_assignments_packed"] = t_p

# Column hypers
t_o, _ = time_fn(transition_column_hypers, key, state, data)
t_p, _ = time_fn(packed_transition_column_hypers, key, packed, data)
print(
    f"  {'column_hypers':25s}  orig: {t_o:.4f}s  packed: {t_p:.4f}s  {t_o / max(t_p, 1e-9):.1f}x"
)
total_orig += t_o
total_packed += t_p
results["column_hypers_orig"] = t_o
results["column_hypers_packed"] = t_p

# CRP alphas
t_o, _ = time_fn(transition_crp_alphas, key, state)
t_p, _ = time_fn(packed_transition_crp_alphas, key, packed)
print(f"  {'crp_alphas':25s}  orig: {t_o:.4f}s  packed: {t_p:.4f}s  {t_o / max(t_p, 1e-9):.1f}x")
total_orig += t_o
total_packed += t_p
results["crp_alphas_orig"] = t_o
results["crp_alphas_packed"] = t_p

print("-" * 70)
speedup = total_orig / max(total_packed, 1e-9)
print(f"  {'TOTAL':25s}  orig: {total_orig:.4f}s  packed: {total_packed:.4f}s  {speedup:.1f}x")
results["total_orig"] = total_orig
results["total_packed"] = total_packed
results["total_speedup"] = speedup

## 4. Full Sweep Benchmark

3 full Gibbs sweeps: unpacked vs packed.

In [ ]:
key = jax.random.key(456)
t_orig_sweep, _ = time_fn(
    gibbs_sweep,
    key,
    state,
    data,
    n_sweeps=3,
    kernels=("row_assignments", "column_hypers", "crp_alphas"),
)
t_packed_sweep, _ = time_fn(packed_gibbs_sweep, key, packed, data, n_sweeps=3)
sweep_speedup = t_orig_sweep / max(t_packed_sweep, 1e-9)
print(
    f"  {'full sweep (3 iters)':30s}  orig: {t_orig_sweep:.4f}s"
    f"  packed: {t_packed_sweep:.4f}s  {sweep_speedup:.1f}x"
)
results["full_sweep_orig"] = t_orig_sweep
results["full_sweep_packed"] = t_packed_sweep
results["full_sweep_speedup"] = sweep_speedup

## 5. Cyclic-Only Benchmark

Tests Von Mises type-specialized fast path with 10 cyclic columns.

In [ ]:
cyclic_types = [ColumnType.CYCLIC] * 10
cyclic_result = generate_crosscat_data(
    jax.random.key(789), 200, cyclic_types, n_views=2, n_clusters=3
)
cyclic_data = cyclic_result["data"]
k_cyc = jax.random.key(790)
cyclic_state = initialize(k_cyc, cyclic_data, cyclic_types).state
cyclic_packed = pack_state(cyclic_state)
t_cyc, _ = time_fn(packed_gibbs_sweep, jax.random.key(791), cyclic_packed, cyclic_data, n_sweeps=3)
print(f"  {'cyclic-only packed (3 iters)':30s}  {t_cyc:.4f}s")
results["cyclic_only_packed"] = t_cyc

## 6. Save Results

In [ ]:
import json
import shutil

results_dir = create_results_dir("jit")
save_metrics(results, results_dir)

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(json.dumps(results, indent=2, default=str))

# Archive for download
archive = Path("benchmarks/results/jit_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"\nArchived to {archive}.tar.gz")